In [5]:
# ================================================================
# ASSIGNMENT 3 - MULTI-SOURCE RETAIL SALES DATA INTEGRATION
# UCI ONLINE RETAIL DATASET
# ================================================================

cat("====================================================\n")
cat("   RETAIL SALES DATA INTEGRATION AND ANALYSIS\n")
cat("====================================================\n\n")


# ------------------------------------------------
# 1. INSTALL AND LOAD REQUIRED PACKAGES
# ------------------------------------------------

packages <- c(
  "readxl",
  "jsonlite",
  "writexl",
  "DBI",
  "RSQLite",
  "dplyr"
)

for (p in packages) {
  if (!requireNamespace(p, quietly = TRUE)) {
    install.packages(p, repos = "https://cloud.r-project.org")
  }
}

library(readxl)
library(jsonlite)
library(writexl)
library(DBI)
library(RSQLite)
library(dplyr)

cat("Packages loaded successfully.\n\n")


# ------------------------------------------------
# 2. DOWNLOAD UCI ONLINE RETAIL DATASET
# ------------------------------------------------

url <- "https://archive.ics.uci.edu/static/public/352/online+retail.zip"

if (!file.exists("online_retail.zip")) {

  cat("Downloading UCI Online Retail Dataset...\n")

  download.file(
    url,
    destfile = "online_retail.zip",
    mode = "wb"
  )

}

if (!file.exists("Online Retail.xlsx")) {

  cat("Extracting dataset...\n")

  unzip(
    "online_retail.zip",
    files = "Online Retail.xlsx",
    overwrite = TRUE
  )

}

cat("Dataset ready.\n\n")


# ------------------------------------------------
# 3. IMPORT ORIGINAL DATASET
# ------------------------------------------------

retail <- read_excel("Online Retail.xlsx")

cat("Original Dataset Information\n")
cat("----------------------------\n")
cat("Rows    :", nrow(retail), "\n")
cat("Columns :", ncol(retail), "\n\n")


# ------------------------------------------------
# 4. DATA INSPECTION
# ------------------------------------------------

cat("Column Names:\n")
print(names(retail))

cat("\nMissing Values:\n")
print(colSums(is.na(retail)))

duplicate_count <- sum(duplicated(retail))

cat("\nDuplicate Records:", duplicate_count, "\n\n")


# ------------------------------------------------
# 5. DATA CLEANING
# ------------------------------------------------

cat("Performing data cleaning...\n")

# Remove duplicate records
retail <- retail %>%
  distinct()

# Remove missing essential fields
retail <- retail %>%
  filter(
    !is.na(InvoiceNo),
    !is.na(StockCode),
    !is.na(Quantity),
    !is.na(UnitPrice)
  )

# Remove zero/negative quantities
retail <- retail %>%
  filter(Quantity > 0)

# Remove zero/negative prices
retail <- retail %>%
  filter(UnitPrice > 0)

cat("Cleaning completed.\n")
cat("Cleaned rows:", nrow(retail), "\n\n")


# ------------------------------------------------
# 6. CREATE REVENUE ATTRIBUTE
# ------------------------------------------------

retail <- retail %>%
  mutate(
    Revenue = Quantity * UnitPrice
  )

cat("Revenue attribute created.\n\n")


# ------------------------------------------------
# 7. CREATE TRANSACTIONS.CSV
# ------------------------------------------------

transactions <- retail %>%
  select(
    InvoiceNo,
    StockCode,
    CustomerID,
    Quantity,
    InvoiceDate
  )

write.csv(
  transactions,
  "transactions.csv",
  row.names = FALSE
)

cat("transactions.csv created.\n")


# ------------------------------------------------
# 8. CREATE PRODUCTS.JSON
# ------------------------------------------------

products <- retail %>%
  select(
    StockCode,
    Description,
    UnitPrice
  ) %>%
  distinct(StockCode, .keep_all = TRUE)

write_json(
  products,
  "products.json",
  pretty = TRUE,
  na = "null"
)

cat("products.json created.\n")


# ------------------------------------------------
# 9. CREATE CUSTOMERS.XLSX
# ------------------------------------------------

customers <- retail %>%
  select(
    CustomerID,
    Country
  ) %>%
  filter(!is.na(CustomerID)) %>%
  distinct(CustomerID, .keep_all = TRUE)

write_xlsx(
  customers,
  "customers.xlsx"
)

cat("customers.xlsx created.\n\n")


# ------------------------------------------------
# 10. IMPORT THREE DATA SOURCES
# ------------------------------------------------

transactions <- read.csv(
  "transactions.csv",
  stringsAsFactors = FALSE
)

products <- fromJSON(
  "products.json"
)

customers <- read_excel(
  "customers.xlsx"
)

cat("All three data sources imported successfully.\n\n")


# ------------------------------------------------
# 11. INTEGRATE DATASETS
# ------------------------------------------------

cat("Integrating datasets...\n")

# Transactions + Products
integrated_data <- transactions %>%
  left_join(
    products,
    by = "StockCode"
  )

# Transactions + Products + Customers
final_data <- integrated_data %>%
  left_join(
    customers,
    by = "CustomerID"
  )

# Calculate Revenue
final_data <- final_data %>%
  mutate(
    Revenue = Quantity * UnitPrice
  )

cat("Integration completed.\n")
cat("Final Rows    :", nrow(final_data), "\n")
cat("Final Columns :", ncol(final_data), "\n\n")


# ------------------------------------------------
# 12. CHECK UNMATCHED RECORDS
# ------------------------------------------------

unmatched_products <- sum(
  is.na(final_data$Description)
)

unmatched_customers <- sum(
  is.na(final_data$Country)
)

cat("Unmatched Product Records :", unmatched_products, "\n")
cat("Unmatched Customer Records:", unmatched_customers, "\n\n")


# ------------------------------------------------
# 13. TOTAL SALES REVENUE
# ------------------------------------------------

total_revenue <- sum(
  final_data$Revenue,
  na.rm = TRUE
)

cat("====================================================\n")
cat("                 SALES ANALYSIS\n")
cat("====================================================\n\n")

cat(
  "TOTAL SALES REVENUE =",
  round(total_revenue, 2),
  "\n\n"
)


# ------------------------------------------------
# 14. TOP 5 PRODUCTS
# ------------------------------------------------

top_products <- final_data %>%
  group_by(
    StockCode,
    Description
  ) %>%
  summarise(
    Revenue = sum(
      Revenue,
      na.rm = TRUE
    ),
    .groups = "drop"
  ) %>%
  arrange(
    desc(Revenue)
  ) %>%
  slice_head(n = 5)

cat("TOP 5 PRODUCTS BY REVENUE\n")
cat("-------------------------\n")
print(top_products)

cat("\n")


# ------------------------------------------------
# 15. TOP 5 COUNTRIES
# ------------------------------------------------

top_countries <- final_data %>%
  group_by(Country) %>%
  summarise(
    Revenue = sum(
      Revenue,
      na.rm = TRUE
    ),
    .groups = "drop"
  ) %>%
  arrange(
    desc(Revenue)
  ) %>%
  slice_head(n = 5)

cat("TOP 5 COUNTRIES BY REVENUE\n")
cat("--------------------------\n")
print(top_countries)

cat("\n")


# ------------------------------------------------
# 16. TOP 5 CUSTOMERS
# ------------------------------------------------

top_customers <- final_data %>%
  filter(
    !is.na(CustomerID)
  ) %>%
  group_by(CustomerID) %>%
  summarise(
    Purchase_Value = sum(
      Revenue,
      na.rm = TRUE
    ),
    .groups = "drop"
  ) %>%
  arrange(
    desc(Purchase_Value)
  ) %>%
  slice_head(n = 5)

cat("TOP 5 CUSTOMERS BY PURCHASE VALUE\n")
cat("---------------------------------\n")
print(top_customers)

cat("\n")


# ------------------------------------------------
# 17. CUSTOMER VALUE CLASSIFICATION
# ------------------------------------------------

customer_value <- final_data %>%
  filter(
    !is.na(CustomerID)
  ) %>%
  group_by(CustomerID) %>%
  summarise(
    Total_Purchase = sum(
      Revenue,
      na.rm = TRUE
    ),
    .groups = "drop"
  ) %>%
  mutate(

    Customer_Category = case_when(

      Total_Purchase < 1000
      ~ "Low Value",

      Total_Purchase < 5000
      ~ "Medium Value",

      Total_Purchase < 10000
      ~ "High Value",

      TRUE
      ~ "Premium"

    )

  )

cat("CUSTOMER VALUE CLASSIFICATION\n")
cat("-----------------------------\n")

print(
  head(customer_value, 10)
)

cat("\n")


# ------------------------------------------------
# 18. CUSTOMER CATEGORY SUMMARY
# ------------------------------------------------

category_summary <- customer_value %>%
  count(
    Customer_Category
  )

cat("CUSTOMER CATEGORY COUNTS\n")
cat("------------------------\n")

print(category_summary)

cat("\n")


# ------------------------------------------------
# 19. MARKET PERFORMANCE
# ------------------------------------------------

market_performance <- final_data %>%
  group_by(Country) %>%
  summarise(
    Revenue = sum(
      Revenue,
      na.rm = TRUE
    ),
    .groups = "drop"
  ) %>%
  arrange(
    desc(Revenue)
  )


# Highest performing market
best_market <- market_performance %>%
  slice_head(n = 1)


# Lowest performing market
underperforming_market <- market_performance %>%
  slice_tail(n = 1)


cat("MARKET PERFORMANCE\n")
cat("------------------\n")

cat("High-performing market:\n")
print(best_market)

cat("\nUnderperforming market:\n")
print(underperforming_market)

cat("\n")


# ------------------------------------------------
# 20. CREATE SQLITE DATABASE
# ------------------------------------------------

cat("Creating SQLite database...\n")

if (file.exists("retail_sales.db")) {
  file.remove("retail_sales.db")
}

con <- dbConnect(
  SQLite(),
  "retail_sales.db"
)


# ------------------------------------------------
# 21. STORE FINAL DATA IN SQLITE
# ------------------------------------------------

dbWriteTable(
  con,
  "retail_sales",
  final_data,
  overwrite = TRUE
)

cat("SQLite table 'retail_sales' created.\n\n")


# ------------------------------------------------
# 22. SQL QUERY 1
# TOP 5 CUSTOMERS
# ------------------------------------------------

sql_query_1 <- "
SELECT
    CustomerID,
    SUM(Revenue) AS Total_Revenue
FROM retail_sales
WHERE CustomerID IS NOT NULL
GROUP BY CustomerID
ORDER BY Total_Revenue DESC
LIMIT 5
"

sql_result_1 <- dbGetQuery(
  con,
  sql_query_1
)

cat("SQL QUERY 1 - TOP 5 CUSTOMERS\n")
cat("-----------------------------\n")

print(sql_result_1)

cat("\n")


# ------------------------------------------------
# 23. SQL QUERY 2
# REVENUE BY COUNTRY
# ------------------------------------------------

sql_query_2 <- "
SELECT
    Country,
    SUM(Revenue) AS Total_Revenue
FROM retail_sales
GROUP BY Country
ORDER BY Total_Revenue DESC
"

sql_result_2 <- dbGetQuery(
  con,
  sql_query_2
)

cat("SQL QUERY 2 - REVENUE BY COUNTRY\n")
cat("--------------------------------\n")

print(
  head(sql_result_2, 10)
)

cat("\n")


# ------------------------------------------------
# 24. DATABASE VERIFICATION
# ------------------------------------------------

cat("DATABASE VERIFICATION\n")
cat("---------------------\n")

print(
  dbListTables(con)
)

database_count <- dbGetQuery(
  con,
  "SELECT COUNT(*) AS Total_Rows FROM retail_sales"
)

cat("\nRows stored in SQLite:\n")
print(database_count)

dbDisconnect(con)


# ------------------------------------------------
# 25. FINAL BUSINESS INSIGHTS
# ------------------------------------------------

cat("\n")
cat("====================================================\n")
cat("                 BUSINESS INSIGHTS\n")
cat("====================================================\n\n")

cat(
  "1. The total sales revenue generated from the cleaned ",
  "retail transactions is ",
  round(total_revenue, 2),
  ".\n\n",
  sep = ""
)

cat(
  "2. The highest-performing market is ",
  as.character(best_market$Country[1]),
  " with revenue of ",
  round(best_market$Revenue[1], 2),
  ".\n\n",
  sep = ""
)

cat(
  "3. Customer purchase values show different customer ",
  "value segments, including Low Value, Medium Value, ",
  "High Value and Premium customers.\n\n"
)


# ------------------------------------------------
# 26. FINAL FILE CHECK
# ------------------------------------------------

cat("====================================================\n")
cat("                 GENERATED FILES\n")
cat("====================================================\n\n")

required_files <- c(
  "transactions.csv",
  "products.json",
  "customers.xlsx",
  "retail_sales.db"
)

for (f in required_files) {

  cat(
    f,
    " --> ",
    ifelse(
      file.exists(f),
      "CREATED",
      "NOT CREATED"
    ),
    "\n"
  )

}

cat("\n====================================================\n")
cat("             ASSIGNMENT COMPLETED\n")
cat("====================================================\n")

   RETAIL SALES DATA INTEGRATION AND ANALYSIS

Packages loaded successfully.

Extracting dataset...
Dataset ready.

Original Dataset Information
----------------------------
Rows    : 541909 
Columns : 8 

Column Names:
[1] "InvoiceNo"   "StockCode"   "Description" "Quantity"    "InvoiceDate"
[6] "UnitPrice"   "CustomerID"  "Country"    

Missing Values:
  InvoiceNo   StockCode Description    Quantity InvoiceDate   UnitPrice 
          0           0        1454           0           0           0 
 CustomerID     Country 
     135080           0 

Duplicate Records: 5268 

Performing data cleaning...
Cleaning completed.
Cleaned rows: 524878 

Revenue attribute created.

transactions.csv created.
products.json created.
customers.xlsx created.

All three data sources imported successfully.

Integrating datasets...
Integration completed.
Final Rows    : 524878 
Final Columns : 9 

Unmatched Product Records : 0 
Unmatched Customer Records: 132186 

                 SALES ANALYSIS

TOTAL SA